# 📚 Libro de análisis · Empleos en CV (mini investigación)

**Datasets de curriculums procesados → ¿qué insight sobre la vida laboral?**

- Este libro responde preguntas de negocio/analíticas sobre **1.506.434 experiencias
  laborales** (de 284.247 personas) del archivo limpio
  `data/cruce/empleos_limpio.parquet`.
- Es **solo lectura**: ninguna celda modifica el parquet ni guarda derivados.
  Toda métrica se **calcula en vivo** y ninguna cifra se escribe a mano.
- Reproduce todo de cero con *Kernel → Restart & Run All*. Las cifras de la
  línea base del bloque 1 se validan contra la auditoría del proyecto
  (`md/AUDITORIA_RAMA_PRUEBA.md`).


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # ejecución fuera de Jupyter (validación automática)
    def display(*objetos, **kwargs):
        for o in objetos:
            if hasattr(o, "to_string"):
                print(o.to_string())
            else:
                print(o)

PROYECTO = Path.cwd()
while not (PROYECTO / "data").exists() and PROYECTO != PROYECTO.parent:
    PROYECTO = PROYECTO.parent
sys.path.insert(0, str(PROYECTO / "script" / "filtro"))
import filtros as F
import indicadores as I

RUTA = PROYECTO / "data" / "cruce" / "empleos_limpio.parquet"
empleos = pd.read_parquet(RUTA)
print("Raíz del proyecto:", PROYECTO)
print("Archivo           :", RUTA.name, f"({empleos.shape[0]:,} filas x {empleos.shape[1]} columnas)")


## 📦 1 · Conocer el dataset

Primero fijamos la **línea base**: si estas cifras no coinciden con la auditoría,
algo cambió en el pipeline y este libro no es interpretable. Luego describimos
periodo, niveles, ocupaciones y banderas de clasificación.


In [ ]:
filas, columnas = empleos.shape
personas = int(empleos["resume_id"].nunique())
linea_base = pd.DataFrame({
    "metrica": [
        "filas", "columnas", "personas", "es_unknown_ocupacion",
        "es_rescatado", "es_vigente", "university_level='No reportado'",
    ],
    "valor": [
        filas, columnas, personas,
        int(empleos["es_unknown_ocupacion"].sum()),
        int(empleos["es_rescatado"].sum()),
        int(empleos["es_vigente"].sum()),
        int(empleos["university_level"].eq("No reportado").sum()),
    ],
})
display(linea_base)

# Validación contra la auditoría: el libro solo tiene sentido si esto pasa.
assert filas == 1_506_434, f"Filas esperadas 1.506.434, se hallaron {filas}"
assert columnas == 14, f"Columnas esperadas 14, se hallaron {columnas}"
assert personas == 284_247, f"Personas esperadas 284.247, se hallaron {personas}"
assert int(empleos["es_unknown_ocupacion"].sum()) == 104_993
assert int(empleos["es_rescatado"].sum()) == 10_176
assert int(empleos["es_vigente"].sum()) == 76_180
assert int(empleos["university_level"].eq("No reportado").sum()) == 174_035
print("Línea base correcta: coincide con la auditoría.")


In [ ]:
inicios = empleos["start_date"].min(), empleos["start_date"].max()
fines = empleos.loc[empleos["end_date"].ne("Present"), "end_date"]
print("Rango de inicios              :", inicios)
print("Rango de fines (sin Present)  :", (fines.min(), fines.max()))
print("Experiencias vigentes ('Present'):", int(empleos["end_date"].eq("Present").sum()))


In [ ]:
niveles = empleos["university_level"].value_counts().to_frame("experiencias")
niveles["personas"] = empleos.groupby("university_level")["resume_id"].nunique()
niveles["%_del_total"] = (niveles["experiencias"] / filas * 100).round(2)
display(niveles)


In [ ]:
ocupaciones = I.top_valores(empleos["occupation_label"], 10)
print(f"Ocupaciones distintas con etiqueta: {int(empleos['occupation_label'].notna().sum()):,} "
      f"(de {empleos['occupation_label'].nunique():,} únicas)")
display(ocupaciones)


In [ ]:
grupos = I.top_valores(empleos["isco_group_label"], 10)
print(f"Grupos ISCO distintos con etiqueta: {empleos['isco_group_label'].nunique():,}")
display(grupos)


In [ ]:
banderas = I.resumen_banderas(empleos)
display(banderas)
sin_area_por_nivel = (
    empleos.loc[empleos["isco_group"].isna(), "university_level"].value_counts()
    / empleos["university_level"].value_counts() * 100
).round(2).sort_values(ascending=False)
print("Sin área ISCO por nivel educativo (% del nivel):")
display(sin_area_por_nivel)


**Qué deja claro el bloque 1:** el dataset cubre más de 6 décadas (inicios Q1 1955–Q4 2020),
en su mayoría experiencia breve (5 trimestres de mediana), con 6 niveles educativos y 426
grupos ocupacionales (ISCO). Las banderas separan lo que sí se sabe de lo que no: `unknown`
(6,97 %), `rescatado` (0,68 %) y `vigente` (5,06 %, final censurado). Las cifras exactas están
en las celdas de arriba; ningún número de este resumen es un invento, todo se calculó ahí.


## 🔎 2 · Exploración dirigida

Cada sección plantea una **pregunta** y la responde con un **filtro reutilizable**
de `script/filtro/filtros.py` o un **indicador** de `script/filtro/indicadores.py`.
Ninguna selección imputa valores: lo no clasificado se excluye y se reporta.


In [ ]:
# P1 · ¿Cuántas personas tienen educación de doctorado y cuántas experiencias?
phd = F.filtrar_por_nivel_educativo(empleos, "PhD")
display(F.resumen(phd, "university_level='PhD'"))


In [ ]:
# P2 · En el grupo ISCO más frecuente, ¿cuántas experiencias/personas hay?
grupo_top = empleos["isco_group_label"].dropna().value_counts().index[0]
sub = F.filtrar_por_grupo_isco(empleos, grupo_top)
print(f"Grupo ISCO más frecuente: {grupo_top}")
display(F.resumen(sub, f"isco_group_label='{grupo_top}'"))


In [ ]:
# P3 · Experiencias iniciadas 2015–2019: ¿cuántas y cuál es su composición?
recientes = F.filtrar_por_periodo(empleos, 2015, 2019)
print(f"Experiencias iniciadas 2015-2019: {len(recientes):,} "
      f"(de {personas} personas, {recientes['resume_id'].nunique():,} aquí)")
print(f"- Vigentes dentro de ese corte: {int(recientes['es_vigente'].sum()):,} "
      f"({100 * recientes['es_vigente'].mean():.1f} % del corte)")
display(F.resumen(recientes, "inicio 2015-2019"))


In [ ]:
# P4 · ¿Vigentes vs finalizadas? (la duración solo existe para las finalizadas)
vigentes = F.filtrar_vigentes(empleos, True)
finalizadas = F.filtrar_vigentes(empleos, False)
print(f"Vigentes  : {len(vigentes):,} ({100 * len(vigentes) / filas:.1f} %)")
print(f"Finalizadas: {len(finalizadas):,} ({100 * len(finalizadas) / filas:.1f} %)")
print(f"Mediana de duración (finalizadas): {I.duracion_trimestres(finalizadas).median():.0f} trimestres")
print("Nota: las vigentes no tienen fin observado; su duración es censura, no 0.")


In [ ]:
# P5 · ¿Cuánto pesa 'sin clasificar' y cómo se reparte por nivel?
sin_area = F.filtrar_sin_area_isco(empleos)
print(f"Experiencias sin área ISCO: {len(sin_area):,} "
      f"({100 * len(sin_area) / filas:.2f} %) · {sin_area['resume_id'].nunique():,} personas")
print("% de experiencias sin área dentro de cada nivel educativo:")
niv_total = empleos["university_level"].value_counts()
niv_sin = sin_area["university_level"].value_counts()
display((niv_sin / niv_total * 100).round(2).rename("%_sin_area").to_frame())


**Qué deja claro el bloque 2:** los filtros responden preguntas concretas y dejan ver el peso
de lo no clasificado: por ejemplo, el grupo ISCO más frecuente concentra decenas de miles de
experiencias, y el porcentaje de experiencias sin área varía entre niveles. Estos números se
usan tal cual en los bloques siguientes.


## 🧩 3 · Relaciones entre variables

Buscamos conexiones entre **nivel educativo**, **grupo ocupacional (ISCO)**, **duración** y
**trayectoria** (transiciones y periodos sin empleo registrado).

> ⚠️ **Pluriempleo y repeticiones:** el parquet limpio conserva experiencias simultáneas
> (misma persona, periodos que se cruzan) y repeticiones exactas. Para transiciones y
> solapamiento las repeticiones exactas se quitan; para brechas, el máximo de fines previos
> absorbe el traslape. Ningún conteo lo inventa el análisis: la definición se declara.


In [ ]:
# Nivel educativo ↔ grupo ISCO (frecuencia relativa por fila-nivel)
tabla_nivel_grupo = I.crosstab_nivel_grupo(empleos, top=30)
display(tabla_nivel_grupo)
# Grupo más representado por nivel
print("Grupos ISCO más representados dentro de cada nivel (top 3, % del nivel):")
for nivel in ["Secondary school", "Bachelor", "Master", "PhD", "No reportado"]:
    vc = empleos.loc[empleos["university_level"].eq(nivel), "isco_group_label"] \
        .dropna().value_counts(normalize=True).head(3) * 100
    print(f"- {nivel}: " + ", ".join(f"{g} {p:.1f}%" for g, p in vc.items()))


In [ ]:
# Grupo ISCO ↔ duración (mediana de trimestres por grupo, top en frecuencia)
duracion_por_grupo = I.duracion_mediana_por_grupo(empleos, top=15)
display(duracion_por_grupo)
print("Grupo con mayor mediana de duración entre los top:",
      duracion_por_grupo["mediana"].idxmax(), "·", duracion_por_grupo["mediana"].max(), "trimestres")


In [ ]:
# Nivel educativo ↔ duración
display(I.duracion_mediana_por_nivel(empleos))


In [ ]:
# Transiciones consecutivas: ¿la gente cambia de grupo ISCO entre experiencias?
trans = I.transiciones_consecutivas(empleos)
mismo = I.proporcion_mismo_grupo(empleos)
print(f"Parejas consecutivas con ambas etiquetas: {int(trans['n'].sum()):,}")
print(f"% que conservan el grupo ISCO: {mismo:.1f} %")
print("Transiciones más frecuentes (desde -> hacia):")
display(trans.head(10))


In [ ]:
# % que conserva el grupo ISCO, por nivel educativo
filas_nivel = []
for nivel in ["Secondary school", "Bachelor", "Master", "PhD", "No reportado"]:
    sub = empleos[empleos["university_level"].eq(nivel)]
    trans_n = I.transiciones_consecutivas(sub)
    filas_nivel.append({
        "nivel": nivel,
        "parejas": int(trans_n["n"].sum()),
        "%_mismo_grupo": round(I.proporcion_mismo_grupo(sub), 2),
    })
tabla_mismo_nivel = pd.DataFrame(filas_nivel)
display(tabla_mismo_nivel)


In [ ]:
# Primera ↔ segunda experiencia (¿dónde aterriza la gente?).
prim_seg = I.primera_a_segunda(empleos)
print(f"Personas con primera y segunda experiencia etiquetadas: "
      f"{int(prim_seg['n'].sum()):,}")
print("Pares primera->segunda más frecuentes:")
display(prim_seg.head(10))


In [ ]:
# Brechas sin empleo registrado (proxy de desempleo / inactividad)
tabla_brechas, resumen_brechas = I.brechas_entre_empleos(empleos)
print("Resumen de brechas (trimestres sin empleo registrado entre experiencias):")
display(pd.DataFrame([resumen_brechas]))
print(f"{resumen_brechas['personas_con_brecha']:,} de {resumen_brechas['personas']:,} personas "
      f"({100 * resumen_brechas['personas_con_brecha'] / resumen_brechas['personas']:.1f} %) "
      f"tienen al menos una brecha; mediana de {resumen_brechas['mediana_trimestres']:.0f} "
      f"trimestres y máximo de {resumen_brechas['max_trimestres']:.0f}.")


In [ ]:
# Brechas por nivel educativo (¿el desempleo es menor con más estudios?)
display(I.brechas_por_nivel(empleos))


In [ ]:
# Traslapes (mismo método del diagnóstico: inicio <= fin de la fila anterior)
traslapes = I.resumen_personas_solapadas(empleos)
display(traslapes)
pct_tras = float(traslapes.loc[traslapes["metrica"].eq("%_personas"), "valor"].iloc[0])
print(f"{pct_tras:.1f} % de las personas tienen al menos un traslape: "
      "se interpreta como pluriempleo real o dato impreciso, no como brecha.")


**Qué deja claro el bloque 3:** el nivel educativo se asocia con el repertorio de grupos ISCO
y con la permanencia: conservar el mismo grupo entre experiencias es minoría (menos de 1 de
cada 5 transiciones), mientras que las brechas sin empleo registrado son mayoritarias (>50 % de
las personas) y parecen menores entre quienes reportan más estudios. Ninguna cifra se escribió a
mano: todas vienen de las celdas de código de este bloque.


## 🧠 4 · Mini investigación

Cuatro preguntas guiadas, cada una con el formato **Pregunta → Filtro/datos → Resultado →
Interpretación**. La última retoma el insight clave del análisis de brechas: el historial no
solo dice *en qué* trabajó cada persona, también *en qué momentos* no había empleo registrado
(proxy de estar o no desempleado en ese lapso).

### Q1 · ¿La concentración ocupacional cambia con el nivel educativo?


In [ ]:
# Pregunta: ¿cambia la concentración de grupos ISCO según el nivel educativo?
# Dato: top 3 grupos por nivel (% de experiencias con etiqueta dentro del nivel).
for nivel in ["Secondary school", "Bachelor", "Master", "PhD", "No reportado"]:
    vc = empleos.loc[empleos["university_level"].eq(nivel), "isco_group_label"] \
        .dropna().value_counts(normalize=True).head(3) * 100
    print(f"{nivel}: " + ", ".join(f"{g} = {p:.1f} %" for g, p in vc.items()))
print()
print("INTERPRETACIÓN: observar si el top de cada nivel es el mismo o cambia el repertorio "
      "ocupacional; un top distinto por nivel sugiere que la educación ordena el destino laboral.")


### Q2 · ¿Cuánto dura una experiencia y cambia por nivel educativo?


In [ ]:
# Dato: distribución global y por nivel.
display(I.distribucion_duracion(empleos))
display(I.duracion_mediana_por_nivel(empleos))
print("INTERPRETACIÓN: mediana e IQR muestran la duración típica; la comparación por nivel "
      "cuantifica si estudiar más prolonga la permanencia en cada empleo.")


### Q3 · ¿Qué tan lineal es la carrera (mismo grupo ISCO entre experiencias)?


In [ ]:
display(tabla_mismo_nivel)
mismo_global = I.proporcion_mismo_grupo(empleos)
print(f"INTERPRETACIÓN: a nivel global solo {mismo_global:.1f} % de las transiciones conservan "
      "el grupo ISCO; por nivel se ve si ese apego crece o cae. Una carrera 'lineal' sería la "
      "excepción, no la regla.")


### Q4 · ¿Cuántos periodos sin empleo registrado hay y cómo cambian por nivel educativo?


In [ ]:
# Dato: brechas por persona y por nivel.
display(pd.DataFrame([resumen_brechas]))
brechas_nivel = I.brechas_por_nivel(empleos)
display(brechas_nivel)
print("INTERPRETACIÓN: la proporción de personas con al menos una brecha y su mediana "
      "aproximan cuánto tiempo pasa alguien sin empleo registrado entre experiencias. "
      "La comparación por nivel sugiere si la educación se asocia con periodos de "
      "desocupación más cortos.")


**Cierre de la mini investigación:** cada pregunta produjo un número con su lectura; las
interpretaciones conectan el dato con la hipótesis de negocio (reclutar/estudiar/la empleabilidad
según nivel). Los resultados exactos quedan en las celdas anteriores.


## 🧭 5 · Hallazgos (con DATO OBSERVADO · INTERPRETACIÓN · LIMITACIÓN)

Cada hallazgo calcula su dato **en vivo** — no hay cifras copiadas.


In [ ]:
# Hallazgo 1 · Concentración ocupacional
top3 = I.top_valores(empleos["occupation_label"], 3)
top3_share = float(top3["%_de_con_etiqueta"].sum())
top1_ocu = str(top3.index[0])
display(top3)
print(f"DATO OBSERVADO: las 3 ocupaciones más frecuentes concentran {top3_share:.1f} % "
      "de las experiencias con etiqueta.")
print(f"INTERPRETACIÓN: el mercado laboral de estos CV se apoya en pocos roles de servicio "
      f"y administrativos (la más común es '{top1_ocu}').")
print("LIMITACIÓN: la etiqueta depende del emparejamiento de la fuente; 6,97 % no tiene "
      "área ISCO y no entra en este total (se reporta por separado).")


In [ ]:
# Hallazgo 2 · Duración típica: experiencias cortas, con cola larga
d = I.distribucion_duracion(empleos)
display(d)
print(f"DATO OBSERVADO: mediana de {d.loc[0, 'mediana']:.0f} trimestres, IQR {d.loc[0, 'q1']:.0f}–"
      f"{d.loc[0, 'q3']:.0f}; {d.loc[0, 'fuera_de_iqr_%']:.0f} % fuera del IQR (Tukey).")
print("INTERPRETACIÓN: la rotación es alta; la cola larga (experiencias que superan por "
      f"mucho la mediana, hasta {d.loc[0, 'max']:.0f} trimestres) son carreras de larga data.")
print("LIMITACIÓN: IQR y mediana se calculan sobre empleos finalizados; las 76.180 vigentes "
      "son censura y no entran.")


In [ ]:
# Hallazgo 3 · La mayoría cambia de grupo ocupacional entre experiencias
m = I.proporcion_mismo_grupo(empleos)
print(f"DATO OBSERVADO: solo {m:.1f} % de las transiciones consecutivas conservan el grupo ISCO.")
print("INTERPRETACIÓN: las trayectorias laborales son poco lineales; cambiar de área es lo común.")
print("LIMITACIÓN: la transición solo se observa entre experiencias etiquetadas (se excluyen "
      "las unknown y las vigentes sin continuidad); el orden usa inicio/fin.")


In [ ]:
# Hallazgo 4 · Periodos sin empleo registrado (desempleo proxy) son la norma
r = resumen_brechas
print(f"DATO OBSERVADO: {r['personas_con_brecha']:,} de {r['personas']:,} personas "
      f"({100 * r['personas_con_brecha'] / r['personas']:.1f} %) tienen ≥1 brecha; "
      f"mediana {r['mediana_trimestres']:.0f} trimestres.")
print("INTERPRETACIÓN: entre experiencias hay lapsos frecuentes sin empleo registrado; el "
      "dato responde a la pregunta de si la persona 'estuvo o no desempleada' en ese tiempo.")
print("LIMITACIÓN: 'sin empleo registrado' ≠ desempleo confirmado: puede ser estudio, "
      "inactividad, trabajo no declarado o vacío del currículum; además 5 % de las filas están "
      "censuradas (Present). Es un proxy, no una medida de desempleo.")


In [ ]:
# Hallazgo 5 · Las fronteras del dato: unknown y censura a mano derecha
display(I.resumen_banderas(empleos))
print("DATO OBSERVADO: 6,97 % sin área ISCO y 5,06 % de experiencias vigentes (fin = Present).")
print("INTERPRETACIÓN: el análisis distintivo depende de que no se impute nada de esto; "
      "la interpretación vive en declarar las exclusiones.")
print("LIMITACIÓN: cualquier conclusión sobre ocupación aplica al 93 % etiquetado, y sobre "
      "duración al 95 % con fin observado.")


### Limitaciones transversales

- **Dato auto-reportado**: los CV no son un registro oficial; la ausencia de un empleo es
  silencio, no certeza de desempleo.
- **Pluriempleo y traslapes**: la definición de "transición" depende de ordenar por inicio/fin;
  el método se declara en cada indicador y puede variar ligeramente frente a otras definiciones.
- **Sin campo de estudio**: responder "¿el empleo se relaciona con lo que estudió?" es solo
  posible por proxy (nivel educativo ↔ grupo ISCO heredado).
- **Censura**: los vigentes no tienen duración; asumimos continuidad solo hasta el corte.

### Reproducibilidad

1. Activar el entorno con `script/requirements/requirements.txt`.
2. Importar el notebook: *Kernel → Restart & Run All*.
3. Filtros e indicadores viven en `script/filtro/` (snake_case, funciones puras y asserts).
4. Las cifras de la línea base deben coincidir con `md/AUDITORIA_RAMA_PRUEBA.md`.
